# 1c — Install native dependencies without sudo

Use this notebook when the A100 Jupyter container has `no new privileges` and cannot run sudo. It installs ImageMagick into `~/stage1-native` using the JupyterHub-provided Mamba executable. The repository's Stage 1 entry points automatically discover this fixed prefix.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

home = Path.home()
mamba = shutil.which("mamba")
ood_python = home / "venv-stage1-ood/bin/python"
prefix = home / "stage1-native"
if not mamba:
    raise SystemExit("STOP: mamba is unavailable")
if not ood_python.exists():
    raise SystemExit(f"STOP: missing {ood_python}; finish notebook 1 first")
print("mamba:", mamba)
print("native prefix:", prefix)


In [ ]:
# Re-running this command is safe; Mamba reuses its package cache.
subprocess.run(
    [mamba, "create", "-y", "-p", str(prefix), "-c", "conda-forge", "imagemagick"],
    check=True,
)
print("ImageMagick installed in", prefix)


In [ ]:
env = os.environ.copy()
env["MAGICK_HOME"] = str(prefix)
env["PATH"] = str(prefix / "bin") + os.pathsep + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = str(prefix / "lib") + os.pathsep + env.get("LD_LIBRARY_PATH", "")
env["MPLBACKEND"] = "Agg"
subprocess.run([str(prefix / "bin/magick"), "-version"], env=env, check=True)
subprocess.run(
    [str(ood_python), "-c", "from wand.api import library; print('MagickWand import OK')"],
    env=env,
    check=True,
)
print("PASS: user-space Stage 1 native dependencies are ready")
print("You may now rerun 02_freeze_design_and_import.ipynb")
